## Heston Volatility
Train a neural network to replicate the mapping from Heston parameters to implied volatility

In [ ]:
import pandas as pd
import numpy as np
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import grad
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from joblib import dump

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load Data

In [ ]:
filename = 'HestonData2.csv'
df = pd.read_csv(filename)
df.head()

In [ ]:
df = df

In [ ]:
y = df['iv'].to_numpy()

In [ ]:
X = df.drop(['price', 'iv'], axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.02, random_state=42)

In [ ]:
standard_scaler = StandardScaler()
X_train = standard_scaler.fit_transform(X_train)
X_test = standard_scaler.transform(X_test)

In [ ]:
dump(standard_scaler, 'heston_scaler.bin', compress=True)

In [ ]:
batch_size = 1024

In [ ]:
train_x = torch.Tensor(X_train).to(device)
train_y = torch.Tensor(y_train).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, 
                                           shuffle=True, drop_last=True)

test_x = torch.Tensor(X_test).to(device)
test_y = torch.Tensor(y_test).to(device)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, 
                                          shuffle=False, drop_last=True)

### Train Model

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []
    grad_errors = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0

        for batch_X, batch_y in train_loader:

            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_errors.append(train_loss)

        # Evaluation on the test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_errors.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} - Train loss: {train_loss:.6f}, Test Loss: {test_loss:.6f}")
    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    return history

In [ ]:
class Swish(nn.Module):
    def __init__(self):
        super(Swish, self).__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, width = 32, use_batch_norm=False):
        super(NeuralNetwork, self).__init__()
        self.use_batch_norm = use_batch_norm
        self.activation = Swish()
        self.fc1 = nn.Linear(7, width)
        self.bn1 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc2 = nn.Linear(width, width)
        self.bn2 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc3 = nn.Linear(width, width)
        self.bn3 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc4 = nn.Linear(width, width)
        self.bn4 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc5 = nn.Linear(width, 1)

    def forward(self, x):
        x = self.activation(self.bn1(self.fc1(x)))
        x = self.activation(self.bn2(self.fc2(x)))
        x = self.activation(self.bn3(self.fc3(x)))
        x = self.activation(self.bn4(self.fc4(x)))
        x = self.fc5(x)
        return x

In [ ]:
no_epochs = 1000
loss_fn = nn.MSELoss()

In [ ]:
hestonnn = NeuralNetwork(256, True).to(device)
optimizer = optim.Adam(hestonnn.parameters(), lr=0.001)
history = train_model(hestonnn, train_loader, test_loader, 
                          loss_fn, optimizer, no_epochs)

In [ ]:
model_path = "hestonnn4.pth"

In [ ]:
torch.save(hestonnn.state_dict(), model_path)

### Plot Losses

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

train_loss_values = history['train_loss']
test_loss_values = history['test_loss']
ax.plot(train_loss_values, color='black', linestyle='-',label = 'training loss', )
ax.plot(test_loss_values, color='black', linestyle='--', label = 'test loss')

ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.legend()
ax.set_ylim(0, 0.001)
fig.savefig('HestonNNLoss4.png', dpi=300, bbox_inches='tight')

### Plot Sample Smiles

In [ ]:
mats = np.round(np.linspace(0.1,5,8),1)
mats

In [ ]:
strikes = np.round(np.linspace(0.7,1.3,11),1)
strikes

In [ ]:
import QuantLib as ql
from py_vollib.black_scholes.implied_volatility import implied_volatility

def hestonIV(S0, r, q, tau, K, V0, lamda, vbar, eta, rho):
    todaysDate = ql.Date(17, 2, 2024)
    ql.Settings.instance().evaluationDate = todaysDate
    maturity = int(tau * 365)
    settlementDate = todaysDate 
    day_count = ql.Actual365Fixed()
    rfr = ql.YieldTermStructureHandle(ql.FlatForward(settlementDate, r, day_count))
    div_yield = ql.YieldTermStructureHandle(ql.FlatForward(settlementDate, q, day_count))
    
    exercise = ql.EuropeanExercise(todaysDate + maturity)
    payoff = ql.PlainVanillaPayoff(ql.Option.Call, K)
    european_option = ql.VanillaOption(payoff, exercise)
        
    spot = ql.QuoteHandle(ql.SimpleQuote(S0))
    heston_process = ql.HestonProcess(rfr, div_yield, spot, V0, lamda, vbar, eta, rho)
    engine = ql.AnalyticHestonEngine(ql.HestonModel(heston_process), 1e-15, int(1e6))
    european_option.setPricingEngine(engine)
    price = european_option.NPV()
    iv = implied_volatility(price, S0, K, tau, r, 'c')
    return iv

In [ ]:
S0 = 1
r = 0
q = 0
eta = 2.5
rho = -0.5
lamda = 5.0
vbar = 0.5
V0 = 0.5

In [ ]:
heston_smile_lst = list()

In [ ]:
for imat in mats:
    smile_lst = list()
    for ik in strikes:
        iv = hestonIV(S0, r, q, imat, ik, V0, lamda, vbar, eta, rho)
        smile_lst.append(iv)
    heston_smile_lst.append(smile_lst)

In [ ]:
heston_smile = np.array(heston_smile_lst)
heston_smile

In [ ]:
heston_input_lst = list()

In [ ]:
for imat in mats:
    for ik in strikes:
        input_lst = list()
        input_lst.append(eta)
        input_lst.append(rho)
        input_lst.append(lamda)
        input_lst.append(vbar)
        input_lst.append(V0)
        input_lst.append(imat)
        input_lst.append(ik)
        heston_input_lst.append(input_lst)

In [ ]:
heading_list = ['eta', 'rho', 'lambda', 'vbar', 'V0', 'tau', 'K']
df = pd.DataFrame(heston_input_lst, columns=heading_list)
df.head()

In [ ]:
heston_X = standard_scaler.transform(df)

In [ ]:
theston_X = torch.Tensor(heston_X).to(device)

In [ ]:
hestonnn_iv = hestonnn(theston_X)

In [ ]:
hestonnn_iv = hestonnn_iv.reshape((8, 11)).detach().cpu().numpy()

In [ ]:
hestonnn_iv

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(14, 10))
axs = axs.flatten()

for i, maturity in enumerate(mats):
    axs[i].plot(strikes, heston_smile[i], color='black', linestyle = '-', label='Heston')
    axs[i].plot(strikes, hestonnn_iv[i], color='black', linestyle = '--', label='DNN')
    axs[i].set_title(f'Maturity = {maturity} years')
    axs[i].set_xlabel('Relative Strike (K/S)')
    axs[i].set_ylabel('Implied Volatility')
    axs[i].legend()

fig.savefig('HestonSmiles4.png', dpi=300, bbox_inches='tight')